In [1]:
import os
import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_arch import RRDB_UNet

In [2]:
pad = 5
device = "cuda"
#device = "cpu"

In [3]:
model = RRDB_UNet(
    num_in_ch=3,
    num_out_ch=3,
    highway_channels_base=28,
    processing_channels_base=16,
    num_grow_ch_base = 6,
    ae_rrdb_blocks=3,
    ae_channel_multipliers = [1,2,4,12,36],
    use_attention=True,
    body_rrdb_blocks = 12
)
_ = model.eval()
_ = model.to(device)

highway_channels_base 28
processing_channels_base 16
num_grow_ch_base 6
ae_rrdb_blocks 3
ae_channel_multipliers [1, 2, 4, 12, 36]
use_attention True
body_rrdb_blocks 12


In [4]:
print(model)

RRDB_UNet(
  (encoder): ModuleList(
    (0): Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01, inplace=True)
    (2-4): 3 x HighwayRRDB(
      (compress): Conv2d(28, 16, kernel_size=(1, 1), stride=(1, 1))
      (rdb1): ResidualDenseBlock(
        (conv1): Conv2d(16, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(22, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv3): Conv2d(28, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv4): Conv2d(34, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv5): Conv2d(40, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (lrelu): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (rdb2): ResidualDenseBlock(
        (conv1): Conv2d(16, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(22, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv3): Conv2d(28, 

In [5]:

def process(img):

    t0 = time.time()
    
    img = img.astype(np.float32)
    if np.max(img) > 256:  # 16-bit image
        max_range = 65535
        print('\tInput is a 16-bit image')
    else:
        max_range = 255
    img = img / max_range
    img = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
    img = img.unsqueeze(0)

    # pre_pad
    img = torch.nn.functional.pad(img, (5, 5, 5, 5), 'reflect').float()
    
    img = img.to(device)
    
    print("testing...")
    with torch.no_grad(): 
        with torch.amp.autocast(device):
            out = model(img)

    # Remove resolution padding first (in reverse order)
    print(out.size())
    
    # remove prepad
    _, _, h, w = out.size()
    out = out[:, :, 5:h - 5, 5:w - 5]
    
    
    output_img = out.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    output_img = np.transpose(output_img[:, :, :], (1, 2, 0))
    
    if max_range == 65535:  # 16-bit image
        output_img = (output_img * 65535.0).round().astype(np.uint16)
    else:
        output_img = (output_img * 255.0).round().astype(np.uint8)

  
    print(time.time() - t0)
    
    return output_img
    


In [6]:
import time
t0 = time.time()

In [7]:
img = cv2.imread("tests/data/lq_4/baboon.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (4050*2, 4050*2))
print(img.shape)
out = process(img)


(8100, 8100, 3)
testing...
torch.Size([1, 3, 8110, 8110])
74.7823052406311


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(out)

In [9]:
time.time() - t0

76.40285038948059